# RL with SAE rewards
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/rl_with_sae_rewards.ipynb)

Use one SAE feature signature as a GRPO reward, then compare feature coverage before and after training.


## Setup
A GPU runtime is recommended: **Runtime → Change runtime type → GPU**. CPU execution is supported but slower. This workflow loads both a trainable model and a frozen SAE host; it may exceed the memory of some Colab runtimes.

Run cells in order. Installation is self-contained; no repository clone or account is needed. If Colab requests a session restart after installation, restart before running the imports. The first model load downloads weights.


In [ ]:
import sys

!"{sys.executable}" -m pip install -q "idiom[cookbook] @ git+https://github.com/rotskoff-group/idiom.git@v1"

In [1]:
import gc
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

from idiom import IDiom, IDiomSAE
from idiom.data.records import Record
from idiom.utils.notebook_helpers import save_run, train_and_save, training_config, write_fasta

started = time.perf_counter()

## Settings
The signature must have been discovered with the same SAE weights used for scoring.


In [2]:
MODEL_ID = "jxliu2/idiom-300M" # Pretrained IDiom model or local release directory
DEVICE = "auto" # "cpu" or "cuda"; "auto" uses an available GPU
BATCH_SIZE = 1 # Prompts per training batch; sequences per inference batch
SEED = 0 # Random seed
MAX_STEPS = 10 # Training steps for this demonstration
TARGET_LENGTH = 50 # Target IDR length in residues
MAX_NEW_TOKENS = 96 # Generation limit, including STOP; not a fixed IDR length
N = 8 # Sequences sampled from each model for comparison
OUT_DIR = Path("rl_sae_outputs") / time.strftime("%Y%m%d-%H%M%S") # New timestamped folder per run

SAE_ID = "jxliu2/idiomsae-300M-L18-k32" # Pretrained SAE and its frozen host model
SAE_DEVICE = "cpu" # CPU scoring saves GPU memory; use "cuda" if memory permits
SIGNATURE_FILE = None # Default: bundled signatures; or a signature.json from the enriched feature signature notebook
CASE = "top30" # Signature group in the JSON file
SIGNATURE_NAME = "nucleolus" # Target signature within CASE
ADDITIONAL_SIGNATURE = None # Optional second signature name; train on the union

In [3]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Select the signature
The reward measures the fraction of target features active anywhere in each IDR, using a frozen host model and SAE.


In [4]:
from idiom.sae.features import load_signatures, write_signature
from idiom.train.grpo.reward.sae_feature import DEFAULT_FEATURES

signature_source = SIGNATURE_FILE or DEFAULT_FEATURES
lens = IDiomSAE.from_pretrained(SAE_ID, device=SAE_DEVICE)
sets = load_signatures(signature_source, case=CASE, sae=SAE_ID, num_latents=lens.sae.num_latents)
requested = [SIGNATURE_NAME] + ([ADDITIONAL_SIGNATURE] if ADDITIONAL_SIGNATURE else [])
missing = [name for name in requested if name not in sets]
if missing:
    raise ValueError(f"Unknown signatures: {missing}. Available names: {sorted(sets)}")
training_signatures = write_signature(
    OUT_DIR / "signature.json",
    {SIGNATURE_NAME: sets[SIGNATURE_NAME]},
    case=CASE,
    provenance=dict(sae=SAE_ID),
)
del lens
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
reward_spec = dict(
    name="sae_signature",
    signature=SIGNATURE_NAME,
    features=str(training_signatures.resolve()),
    case=CASE,
    sae=SAE_ID,
    device=SAE_DEVICE,
)
print(
    f"Target: {SIGNATURE_NAME!r}, {len(sets[SIGNATURE_NAME])} features; first IDs: {sets[SIGNATURE_NAME][:5]}"
)

Target: 'nucleolus', 30 features; first IDs: [6459, 6832, 5813, 2818, 2732]


## Optional variant: combine signatures
The union removes duplicate feature IDs. A high union score can hide low coverage of one component, so inspect component scores after training.


In [5]:
from idiom.sae.features import combine_signatures

component_names = (
    list(dict.fromkeys([SIGNATURE_NAME, ADDITIONAL_SIGNATURE])) if ADDITIONAL_SIGNATURE else [SIGNATURE_NAME]
)
if ADDITIONAL_SIGNATURE:
    if "combined" in component_names:
        raise ValueError("The name 'combined' is reserved for the union in this example.")
    components = {name: sets[name] for name in component_names}
    combined = combine_signatures(components)
    training_signatures = write_signature(
        OUT_DIR / "signature.json",
        {**components, "combined": combined},
        case=CASE,
        provenance=dict(sae=SAE_ID, components=component_names),
    )
    reward_spec["signature"] = "combined"
    print(f"Union: {len(combined)} features; first IDs: {combined[:5]}")

## Sample the starting model


In [6]:
base = IDiom.from_pretrained(MODEL_ID, device=DEVICE)
sampling = dict(n=N, batch_size=BATCH_SIZE, seed=SEED, temperature=1.0, max_new_tokens=MAX_NEW_TOKENS)
baseline = base.generate_unprompted(**sampling)
write_fasta(
    [Record(f"baseline_{i}", s, 0, len(s)) for i, s in enumerate(baseline) if s],
    OUT_DIR / "baseline.fasta",
)
del base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Train
Run a small demonstration job. Training saves a configuration, CSV metrics, a checkpoint, and a reloadable model in `OUT_DIR`. Logging stays local; no account or API key is required.


In [7]:
cfg = training_config("grpo", MODEL_ID, OUT_DIR, device=DEVICE, steps=MAX_STEPS, seed=SEED)
cfg.prompts.n = max(16, MAX_STEPS * BATCH_SIZE)
cfg.prompts.batch_size = BATCH_SIZE
cfg.grpo.group_size = 2
cfg.grpo.max_new_tokens = MAX_NEW_TOKENS
cfg.grpo.lr = 5e-6
cfg.grpo.track_disorder = False
cfg.grpo.log_samples_every = 0
terms = [
    dict(
        label="length",
        weight=1.0,
        reward=dict(name="length"),
        shaping=dict(name="quadratic", target=TARGET_LENGTH, width=0.2),
    ),
    dict(
        label="entropy",
        weight=1.0,
        reward=dict(name="entropy"),
        shaping=dict(name="quadratic", target=3.65, width=0.2),
    ),
]

terms.append(dict(label="signature", weight=1.0, reward=reward_spec, shaping=dict(name="identity")))

cfg.reward.terms = terms
release = train_and_save(cfg)

## Inspect training
Inspect the latest logged metrics. Full history is in `training/metrics/`. Blank entries indicate a metric was not logged on that step.


In [8]:
metrics_path = next((OUT_DIR / "training/metrics").glob("version_*/metrics.csv"))
metrics = pd.read_csv(metrics_path)
columns = ["step"] + [name for name in ["train/loss", "train/reward", "train/kl"] if name in metrics]
print(f"Logged {len(metrics)} metric rows; latest values:")
display(metrics[columns].tail(3))

Logged 10 metric rows; latest values:


,step,train/loss,train/reward,train/kl
7,7,0.115047,-0.786797,0.000142
8,8,0.400002,-7.026205,0.000099
9,9,0.185187,-11.891168,0.000068


## Generate from the saved model
Use the same sampling settings as the baseline for the comparison.


In [9]:
adapted_model = IDiom.from_pretrained(release, device=DEVICE)
adapted = adapted_model.generate_unprompted(**sampling)
write_fasta(
    [Record(f"adapted_{i}", s, 0, len(s)) for i, s in enumerate(adapted) if s],
    OUT_DIR / "adapted.fasta",
)
del adapted_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Inspect scores
Compare raw scores and the total shaped reward. Short demonstration runs may not improve the objective and do not establish sequence function. Full per-sequence scores are saved in `rewards.csv`.


In [10]:
from idiom.train.grpo.reward import build_reward

objective = build_reward(cfg.reward)
rows = []
for group, sequences in (("baseline", baseline), ("adapted", adapted)):
    totals, details = objective(sequences, group_size=1)
    scores = pd.DataFrame(details)
    scores["total_reward"] = totals
    scores["sequence"] = sequences
    scores["group"] = group
    rows.append(scores)
comparison = pd.concat(rows, ignore_index=True)
comparison.to_csv(OUT_DIR / "rewards.csv", index=False)
with pd.option_context("display.max_colwidth", 80):
    display(comparison[["group", "sequence", "total_reward"]].groupby("group", sort=False).head(2))

summary = comparison.groupby("group", sort=False).mean(numeric_only=True)
summary.to_csv(OUT_DIR / "reward_summary.csv")
score_columns = [f"{term['label']}_raw" for term in terms] + ["total_reward"]
display(summary[score_columns].round(3))

,group,sequence,total_reward
0,baseline,DKKEWGNEAEHEASGGRPHAPPQASSIQVL,-4.004336
1,baseline,SSRKKRRKPAAADPPPYDSSKRVKVDGPAGDFSA,-2.827489
8,adapted,DKKEWGNEAEHEASGGRPHAPPQASSIQVL,-4.004336
9,adapted,SSRKKRRKPAAADEPWYDSSDEEAVDGGAGDFGAESGPDEGVHVDATDYRPAESGDSDEQGVSVPDDDYEDAEAARTRS,-8.160858


,length_raw,entropy_raw,signature_raw,total_reward
group,,,,
baseline,43.875,3.568,0.046,-2.597
adapted,51.250,3.610,0.075,-5.626


In [11]:
if ADDITIONAL_SIGNATURE:
    from idiom.train.grpo.reward.sae_feature import sae_signature

    coverage_rows = []
    for name in component_names + ["combined"]:
        score = sae_signature(
            name,
            features=str(training_signatures.resolve()),
            case=CASE,
            sae=SAE_ID,
            device=SAE_DEVICE,
        )
        for group, sequences in (("baseline", baseline), ("adapted", adapted)):
            coverage_rows.extend(
                dict(group=group, signature=name, sequence_id=i, coverage=value)
                for i, value in enumerate(score(sequences))
            )
    coverage = pd.DataFrame(coverage_rows)
    coverage.to_csv(OUT_DIR / "signature_coverage.csv", index=False)
    display(coverage.groupby(["group", "signature"]).coverage.mean().unstack())

## Results
`baseline.fasta`, `adapted.fasta`, and `rewards.csv` contain the before/after samples and scores. `signature.json` records the targets; `signature_coverage.csv` compares component coverage when signatures are combined. `model/` and `training/` contain the release, logs, and checkpoint.

`run.json` records the settings and package versions. Open the output folder in Colab’s **Files** pane to download results. Download them before the runtime ends, or copy them to mounted Drive. Saved notebook previews do not include the exported files.


Saved previews show a CPU example run. Run the cells to create the exported files.

In [12]:
save_run(
    OUT_DIR,
    dict(model=MODEL_ID, device=DEVICE, seed=SEED, steps=MAX_STEPS, sampling=sampling),
    elapsed=time.perf_counter() - started,
)
print("Results folder:", OUT_DIR)

Results folder: rl_sae_outputs/20260921-134326
